Inspect CMIP6 NetCDF Dataset

In [ ]:
from pathlib import Path
import xarray as xr

# 1. Open dataset
nc_path = Path(r"C:\Users\Win 11 New\Desktop\pa gk\datasets\climate\raw\Klima_Daten_10m_3h_2020_RCP26.nc")

ds = xr.open_dataset(nc_path, decode_times=False)

# 2. Dataset overview
print("=== CMIP6 DATASET OVERVIEW ===")
print(ds)

# 3. Dimensions
print("\n=== DIMENSIONS ===")
for dim, size in ds.sizes.items():
    print(f"{dim}: {size}")

# 4. Coordinates
print("\n=== COORDINATES ===")
for name, coord in ds.coords.items():
    print(
        f"{name}: shape={coord.shape}, "
        f"dtype={coord.dtype}, "
        f"units={coord.attrs.get('units', 'N/A')}"
    )

# 5. Climate variables
print("\n=== DATA VARIABLES ===")
for name, var in ds.data_vars.items():
    print(f"\nVariable: {name}")
    print("Shape:", var.shape)
    print("Dimensions:", var.dims)
    print("Data type:", var.dtype)
    print("Long name:", var.attrs.get("long_name", "N/A"))
    print("Units:", var.attrs.get("units", "N/A"))

# 6. Global attributes
print("\n=== GLOBAL ATTRIBUTES ===")
for key, value in ds.attrs.items():
    print(f"{key}: {value}")

ds.close()


=== CMIP6 DATASET OVERVIEW ===
<xarray.Dataset> Size: 2GB
Dimensions:       (rlon: 424, rlat: 412, time: 2928, vertices: 4)
Coordinates:
  * rlon          (rlon) float64 3kB -28.38 -28.26 -28.16 ... 17.93 18.05 18.16
  * rlat          (rlat) float64 3kB -23.38 -23.26 -23.16 ... 21.61 21.73 21.83
  * time          (time) float64 23kB 2.56e+04 2.56e+04 ... 2.596e+04 2.596e+04
    lat           (rlat, rlon) float64 1MB ...
    lon           (rlat, rlon) float64 1MB ...
    height        float64 8B ...
Dimensions without coordinates: vertices
Data variables:
    rotated_pole  |S1 1B ...
    sfcWind       (time, rlat, rlon) float32 2GB ...
    lon_vertices  (rlat, rlon, vertices) float64 6MB ...
    lat_vertices  (rlat, rlon, vertices) float64 6MB ...
Attributes: (12/23)
    contact:                        cordex-clm@dkrz.de, Klaus Keuler (keuler@...
    CORDEX_domain:                  EUR-11
    institute_id:                   CLMcom-BTU
    institution:                    Chair of Environ

Inspect Time and Spatial Coordinates

In [6]:
from netCDF4 import num2date

ds = xr.open_dataset(nc_path, decode_times=False)

# 1. Time information
time = ds["time"]
time_units = time.attrs["units"]
calendar = time.attrs.get("calendar", "standard")

dates = num2date(
    [time.values[0], time.values[-1]],
    units=time_units,
    calendar=calendar
)

print("=== TIME INFORMATION ===")
print("Start:", dates[0])
print("End:", dates[1])
print("Number of timestamps:", ds.sizes["time"])
print("Time interval (hours):", (time.values[1] - time.values[0]) * 24)

# 2. Spatial information
lat = ds["lat"]
lon = ds["lon"]

print("\n=== SPATIAL INFORMATION ===")
print("Grid shape:", lat.shape)
print("Latitude range:", float(lat.min()), "to", float(lat.max()))
print("Longitude range:", float(lon.min()), "to", float(lon.max()))
print("Number of grid points:", ds.sizes["rlat"] * ds.sizes["rlon"])

# 3. Wind speed metadata
wind = ds["sfcWind"]

print("\n=== WIND SPEED ===")
print("Shape:", wind.shape)
print("Units:", wind.attrs.get("units"))
print("Height:", ds["height"].values.item())
print("Height units:", ds["height"].attrs.get("units"))

ds.close()


=== TIME INFORMATION ===
Start: 2020-01-01 00:00:00
End: 2020-12-31 21:00:00
Number of timestamps: 2928
Time interval (hours): 3.0

=== SPATIAL INFORMATION ===
Grid shape: (412, 424)
Latitude range: 21.987829208374023 to 72.58499908447266
Longitude range: -44.59386444091797 to 64.9643783569336
Number of grid points: 174688

=== WIND SPEED ===
Shape: (2928, 412, 424)
Units: m s-1
Height: 10.0
Height units: m


Germany Grid and Rotated Coordinates

In [10]:

import numpy as np
from scipy.spatial import cKDTree

# 1. Coordinate transformation
def rotate_coordinates(lat, lon, pole_lat, pole_lon):
    lat, lon = np.deg2rad(lat), np.deg2rad(lon)
    pole_lat, pole_lon = np.deg2rad(pole_lat), np.deg2rad(pole_lon)

    lon = lon - pole_lon

    rot_lat = np.arcsin(
        np.sin(lat) * np.sin(pole_lat)
        + np.cos(lat) * np.cos(pole_lat) * np.cos(lon)
    )

    rot_lon = np.arctan2(
        np.cos(lat) * np.sin(lon),
        np.sin(lat) * np.cos(pole_lat)
        - np.cos(lat) * np.sin(pole_lat) * np.cos(lon)
    )

    return np.rad2deg(rot_lat), np.rad2deg(rot_lon)


# 2. Germany geographic grid
lats = np.arange(47.3, 55.1, 0.05)
lons = np.arange(5.9, 15.0, 0.05)

target_lats, target_lons = np.meshgrid(lats, lons)
target_lats = target_lats.ravel()
target_lons = target_lons.ravel()

# 3. Map geographic coordinates to NetCDF grid
def to_cartesian(lat, lon):
    lat = np.deg2rad(lat)
    lon = np.deg2rad(lon)

    return np.column_stack((
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ))

with xr.open_dataset(nc_path, decode_times=False) as ds:
    grid_lat = ds["lat"].values.ravel()
    grid_lon = ds["lon"].values.ravel()

    points = to_cartesian(grid_lat, grid_lon)
    targets = to_cartesian(target_lats, target_lons)

    tree = cKDTree(points)
    distances, grid_indices = tree.query(targets)

    source_i, source_j = np.unravel_index(
        grid_indices, ds["lat"].shape
    )

# 4. Figure 4 spatial indices
location_indices = [0, 5000, 10000, 15000, 20000]
print("=== CORRECTED LOCATION MAPPING ===")
for idx in location_indices:
    print(
        f"Location {idx:5d}: "
        f"Target=({target_lats[idx]:.4f}, {target_lons[idx]:.4f}), "
        f"Actual=({grid_lat[grid_indices[idx]]:.4f}, "
        f"{grid_lon[grid_indices[idx]]:.4f})"
    )


=== CORRECTED LOCATION MAPPING ===
Location     0: Target=(47.3000, 5.9000), Actual=(47.2835, 5.9745)
Location  5000: Target=(53.9500, 7.4500), Actual=(53.9985, 7.5121)
Location 10000: Target=(52.7500, 9.0500), Actual=(52.7902, 9.0800)
Location 15000: Target=(51.5500, 10.6500), Actual=(51.5630, 10.5647)
Location 20000: Target=(50.3500, 12.2500), Actual=(50.3357, 12.3153)


Extract Wind Speed Time Series

In [11]:
location_indices = [0, 5000, 10000, 15000, 20000]
wind_series = []

# 1. Extract wind speed at selected locations
with xr.open_dataset(nc_path, decode_times=False) as ds:
    for idx in location_indices:
        i = int(source_i[idx])
        j = int(source_j[idx])

        actual_lat = float(ds["lat"].isel(rlat=i, rlon=j))
        actual_lon = float(ds["lon"].isel(rlat=i, rlon=j))

        print(
            f"Location {idx:5d}: "
            f"Target=({target_lats[idx]:.4f}, {target_lons[idx]:.4f}), "
            f"Actual=({actual_lat:.4f}, {actual_lon:.4f})"
        )

        series = ds["sfcWind"].isel(
            rlat=i,
            rlon=j
        ).values.astype(np.float32)

        wind_series.append(series)

# 2. Combine time series
wind_raw = np.column_stack(wind_series)

# 3. Check extracted data
print("\n=== WIND SPEED DATA ===")
print("Shape:", wind_raw.shape)
print("Data type:", wind_raw.dtype)
print("Missing values:", np.isnan(wind_raw).sum())
print("Infinite values:", np.isinf(wind_raw).sum())
print("Minimum:", np.nanmin(wind_raw))
print("Maximum:", np.nanmax(wind_raw))

print("\nFirst 5 observations:")
print(wind_raw[:5])


Location     0: Target=(47.3000, 5.9000), Actual=(47.2835, 5.9745)
Location  5000: Target=(53.9500, 7.4500), Actual=(53.9985, 7.5121)
Location 10000: Target=(52.7500, 9.0500), Actual=(52.7902, 9.0800)
Location 15000: Target=(51.5500, 10.6500), Actual=(51.5630, 10.5647)
Location 20000: Target=(50.3500, 12.2500), Actual=(50.3357, 12.3153)

=== WIND SPEED DATA ===
Shape: (2928, 5)
Data type: float32
Missing values: 0
Infinite values: 0
Minimum: 0.02670047
Maximum: 23.064035

First 5 observations:
[[1.138077   4.102069   0.9765581  1.2122504  2.6071727 ]
 [0.44310927 2.7986221  0.9172874  0.864127   2.4495466 ]
 [1.300806   2.9371698  1.71782    1.2497466  2.7001398 ]
 [0.19893278 3.0117416  1.9249883  0.9192054  2.513055  ]
 [1.4373713  4.6838465  2.2188601  1.2293766  2.6096117 ]]


Min-Max Scaling and Sliding Windows

In [12]:

import torch
from sklearn.preprocessing import MinMaxScaler

patch_len = 80
horizon = 10

# 1. Select data
sub = wind_raw[:400, :]

# 2. Min-Max Scaling
scaler = MinMaxScaler().fit(sub)
sub = scaler.transform(sub)

# 3. Sliding windows
X, Y = [], []

for i in range(len(sub) - patch_len - horizon):
    X.append(sub[i:i + patch_len])
    y = sub[i + patch_len:i + patch_len + horizon].T.reshape(-1)
    Y.append(y)

# 4. Convert to PyTorch tensors
x_in = torch.tensor(np.stack(X), dtype=torch.float32)
x_out = torch.tensor(np.stack(Y), dtype=torch.float32)

print("=== PREPROCESSED DATA ===")
print("Selected data:", sub.shape)
print("Normalized range:", sub.min(), sub.max())
print("X shape:", x_in.shape)
print("Y shape:", x_out.shape)
print("Number of samples:", len(x_in))


=== PREPROCESSED DATA ===
Selected data: (400, 5)
Normalized range: 0.0 1.0
X shape: torch.Size([310, 80, 5])
Y shape: torch.Size([310, 50])
Number of samples: 310


In [13]:
experiment_dir = Path(nc_path).parents[3] / "experiments" / "02_climate"
results_dir = experiment_dir / "data"
results_dir.mkdir(parents=True, exist_ok=True)

dataset_path = results_dir / "wind_nearest_5sites.pt"

torch.save({
    "wind_raw": torch.from_numpy(wind_raw.copy()),
    "x_in": x_in.cpu(),
    "x_out": x_out.cpu(),
    "location_indices": location_indices,
    "source_i": source_i[location_indices].tolist(),
    "source_j": source_j[location_indices].tolist(),
    "patch_len": patch_len,
    "horizon": horizon,
    "n_time_used": 400,
    "sampling_method": "nearest_neighbor",
    "source_file": Path(nc_path).name
}, dataset_path)

print("Dataset saved:", dataset_path)
print("X:", x_in.shape)
print("Y:", x_out.shape)


Dataset saved: C:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\data\wind_nearest_5sites.pt
X: torch.Size([310, 80, 5])
Y: torch.Size([310, 50])
